In [10]:
import zrp
from zrp.prepare import ZGeo
import pandas as pd
import numpy as np

mapping = {
    "FIRST_NAME": "first_name", 
    "MIDDLE_NAME": "middle_name", 
    "LAST_NAME": "last_name", 
    "HOUSE_NUMBER": "house_number", 
    "STREET_ADDRESS": "street_address", 
    "CITY": "city", 
    "STATE": "state", 
    "ZIP_CODE": "zip_code",
    "RECID": "ZEST_KEY",
}

df = pd.read_parquet("data/temp.parquet")

df["ADDRESS"] = (
    df["ADDRESS"]
    .astype(str)
    .str.replace(r"[^\x00-\x7F]+", "", regex=True)
)

df["address_split"] = df["ADDRESS"].str.split(" ")

df["HOUSE_NUMBER"] = (
    df["address_split"]
    .str[0]
    .str.strip()
)

# keep only numeric house numbers
df["HOUSE_NUMBER"] = (
    pd.to_numeric(df["HOUSE_NUMBER"], errors="coerce")
    .astype("Int64")
    .astype("string")
)

df["STREET_ADDRESS"] = np.where(
    df["HOUSE_NUMBER"].isna(),
    df["ADDRESS"],
    df["address_split"].str[1:].str.join(" ")
)

df["ZIP_CODE"] = df["ZIP_CODE"].astype("string")
df["RECID"] = df["RECID"].astype("string")

for c in ("CITY", "STATE", "ZIP_CODE", "HOUSE_NUMBER", "STREET_ADDRESS"):
    df[c] = df[c].replace("", pd.NA)
    
df = df.rename(columns=mapping)

df = df[list(mapping.values())]

mask = ~(
    df["house_number"].isna()
    & df["city"].isna()
    & df["state"].isna()
    & (
        df["street_address"].isna()
        | (df["street_address"] == "CONFIDENTIAL")
    )
)

df = df.loc[mask].reset_index(drop=True)

df.shape

(10000, 9)

In [24]:
# Run input data through ZRP
zest_race_predictor = zrp.ZRP(runname="temp")
output = zest_race_predictor.transform(df)

zrp_results = pd.read_parquet("artifacts/Zest_Geocoded_temp__2019__37_1.parquet")
zrp_results

####################################
Processing rows: 0:25000
####################################
Data is loaded
   [Start] Validating input data
     Number of observations: 10000
     Is key unique: True
Directory already exists
   [Completed] Validating input data

   Formatting P1


  0%|          | 0/1 [00:00<?, ?it/s]

   Formatting P2
   reduce whitespace

[Start] Preparing geo data

  The following states are included in the data: ['NC']
   ... on state: NC

   Data is loaded
   [Start] Processing geo data
      ...address cleaning


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 180 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 430 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 780 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 1230 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 1780 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 2430 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 3180 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 4030 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 4980 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 6030 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done 7180 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done 8430 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done 9780 tasks      | elapsed:    0.3s
100%|██████████| 10000/10000 [00:00<00:0

      ...replicating address
         ...Base
         ...Number processing...
         House number dataframe expansion is complete! (n=10014)
         ...Base
         ...Map street suffixes...
         ...Mapped & split by street suffixes...
         ...Number processing...

         Address dataframe expansion is complete! (n=13163)
      ...formatting
   [Completed] Processing geo data
   [Start] Mapping geo data
      ...merge user input & lookup table
      ...mapping


...

   [Completed] Validating input geo data
Directory already exists
...Output saved
   [Completed] Mapping geo data

[Completed] Preparing geo data

[Start] Preparing ACS data
   [Start] Validating ACS input data
     Number of observations: 10000
     Is key unique: True

   [Completed] Validating ACS input data

   ...loading ACS lookup tables


   ... combining ACS & user input data
 ...Copy dataframes
 ...Block group
 ...Census tract
 ...Zip code
 ...No match
 ...Merge
 ...Merging complete
[Complete] Preparing ACS data

   [Start] Validating pipeline input data
     Number of observations: 34021
     Is key unique: False
   [Completed] Validating pipeline input data



100%|██████████| 1/1 [00:00<00:00, 917.39it/s]
[Parallel(n_jobs=-1)]: Done   1 out of   1 | elapsed:    0.0s finished
100%|██████████| 1/1 [00:00<00:00, 3368.92it/s]
[Parallel(n_jobs=-1)]: Done   1 out of   1 | elapsed:    0.0s finished
100%|██████████| 1/1 [00:00<00:00, 3650.40it/s]
[Parallel(n_jobs=-1)]: Done   1 out of   1 | elapsed:    0.0s finished


   ...Proxies generated
...Output saved
...Output saved


,first_name,middle_name,last_name,house_number,street_address,city,state,zip_code,zest_in_state_fips,ZEST_KEY_COL,FROMHN_LEFT,TOHN_LEFT,GEOID_CT,GEOID_BG,GEOID_ZIP,GEOID
ZEST_KEY,,,,,,,,,,,,,,,,
10008,STORMY,GAIL,BLACKWELL,1974,WEBSTER GROVE DR,MEBANE,NC,27302,37,10008,,,37001021204,370010212041,27302,None
10009,SUE,RICKS,BLACKWELL,774,BAKER CT,HAW RIVER,NC,27258,37,10009,,,37001021206,370010212061,27258,None
10010,SUSAN,DENISE,BLACKWELL,142,MONTREE LN,GRAHAM,NC,27253,37,10010,,,37001021102,370010211021,27253,None
10025,TYRONE,CLIFTON,BLACKWELL,235,SCHOOL ST,BURLINGTON,NC,27215,37,10025,,,None,None,27215,None
10029,WILLIAM,GREGORY,BLACKWELL,1739,PETTY RD,GRAHAM,NC,27253,37,10029,,,37105030601,371050306011,27253,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9965,LUCILLE,BAKER,BLACKWELL,656,PLAID ST,BURLINGTON,NC,27215,37,9965,,,37001020100,370010201001,27215,None
9971,MARY,LOU,BLACKWELL,4952,BAKER BELL FARM RD,BURLINGTON,NC,27217,37,9971,,,37001021500,370010215002,27217,None
9972,MCCOTTA,ELIZABETH,BLACKWELL,531,E WILLOWBROOK DR,BURLINGTON,NC,27215,37,9972,,,37001020100,370010201001,27215,None


In [5]:
# Modify df for cencus API
df["street"] = (
	df["house_number"].fillna("").str.strip()
    + " "
    + df["street_address"].fillna("").str.strip()
).str.strip()

batch_df = df[[
    "ZEST_KEY",
    "street",
    "city",
    "state",
    "zip_code"
]]

batch_df = batch_df.rename(columns={"zip_code": "zip"})

# Trimming df due to batch size limit
batch_df = batch_df.iloc[:9900]

batch_df.to_csv(
    "census_batch.csv",
    index=False,
    header=True
)


In [6]:
import requests

url = "https://geocoding.geo.census.gov/geocoder/geographies/addressbatch"

files = {
    "addressFile": open("census_batch.csv", "rb")
}

data = {
    "benchmark": "Public_AR_Current",
    "vintage": "Current_Current"
}

response = requests.post(url, files=files, data=data)

with open("census_batch_results.csv", "wb") as f:
    f.write(response.content)

In [7]:
zrp_results

,first_name,middle_name,last_name,house_number,street_address,city,state,zip_code,zest_in_state_fips,ZEST_KEY_COL,FROMHN_LEFT,TOHN_LEFT,GEOID_CT,GEOID_BG,GEOID_ZIP,GEOID
ZEST_KEY,,,,,,,,,,,,,,,,
10008,STORMY,GAIL,BLACKWELL,1974,WEBSTER GROVE DR,MEBANE,NC,27302,37,10008,,,37001021204,370010212041,27302,None
10009,SUE,RICKS,BLACKWELL,774,BAKER CT,HAW RIVER,NC,27258,37,10009,,,37001021206,370010212061,27258,None
10010,SUSAN,DENISE,BLACKWELL,142,MONTREE LN,GRAHAM,NC,27253,37,10010,,,37001021102,370010211021,27253,None
10025,TYRONE,CLIFTON,BLACKWELL,235,SCHOOL ST,BURLINGTON,NC,27215,37,10025,,,None,None,27215,None
10029,WILLIAM,GREGORY,BLACKWELL,1739,PETTY RD,GRAHAM,NC,27253,37,10029,,,37105030601,371050306011,27253,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9965,LUCILLE,BAKER,BLACKWELL,656,PLAID ST,BURLINGTON,NC,27215,37,9965,,,37001020100,370010201001,27215,None
9971,MARY,LOU,BLACKWELL,4952,BAKER BELL FARM RD,BURLINGTON,NC,27217,37,9971,,,37001021500,370010215002,27217,None
9972,MCCOTTA,ELIZABETH,BLACKWELL,531,E WILLOWBROOK DR,BURLINGTON,NC,27215,37,9972,,,37001020100,370010201001,27215,None


In [13]:
# reads batch results to a dataframe
cols = [
    "ZEST_KEY",
    "input_address",
    "match_status",
    "match_type",
    "matched_address",
    "coordinates",
    "tiger_line_id",
    "side",
    "state_fips",
    "county_fips",
    "tract",
    "block",
]

df_census = pd.read_csv(
    "census_batch_results.csv",
    header=None,
    names=cols,
    dtype=str
)

df_census["GEOID_CT"] = (
    df_census["state_fips"]
    + df_census["county_fips"]
    + df_census["tract"]
)

df_census["GEOID_BG"] = (
    df_census["GEOID_CT"]
    + df_census["block"].str[0]
)

df_census.columns

Index(['ZEST_KEY', 'input_address', 'match_status', 'match_type',
       'matched_address', 'coordinates', 'tiger_line_id', 'side', 'state_fips',
       'county_fips', 'tract', 'block', 'GEOID_CT', 'GEOID_BG'],
      dtype='object')

In [22]:
merged = zrp_results.merge(
    df_census[["ZEST_KEY", "GEOID_CT", "GEOID_BG", "state_fips", "county_fips", "tract"]],
    on="ZEST_KEY",
    suffixes=("_zrp", "_census")
)

# drop failed entries
merged = merged.dropna(subset=["GEOID_CT_zrp", "GEOID_CT_census"])

merged["CT_match"] = merged["GEOID_CT_zrp"] == merged["GEOID_CT_census"]
print("CT Id match: ")
print(merged["CT_match"].value_counts(normalize=True))

merged["ct_state_match"] = merged["state_fips"] == merged["GEOID_CT_zrp"].str[:2]
merged["ct_county_match"] = merged["county_fips"] == merged["GEOID_CT_zrp"].str[2:5]
merged["ct_tract_match"] = merged["tract"] == merged["GEOID_CT_zrp"].str[5:]

print("States match: ")
print(merged["ct_state_match"].value_counts(normalize=True))
print("Counties match match: ")
print(merged["ct_county_match"].value_counts(normalize=True))
print("Tracts match: ")
print(merged["ct_tract_match"].value_counts(normalize=True))

print(merged.columns)
print(merged["CT_match"].value_counts(normalize=True))

merged.groupby("ct_state_match")["ct_county_match"].value_counts(normalize=True)
merged[['house_number', 'street_address', 'city', 'state', 'zip_code', 'GEOID_CT_census', 'GEOID_CT_zrp', 'ct_state_match', 'ct_county_match', 'ct_tract_match']].head(n=10)


CT Id match: 
True     0.513458
False    0.486542
Name: CT_match, dtype: float64
States match: 
True    1.0
Name: ct_state_match, dtype: float64
Counties match match: 
True     0.658467
False    0.341533
Name: ct_county_match, dtype: float64
Tracts match: 
True     0.518609
False    0.481391
Name: ct_tract_match, dtype: float64
Index(['ZEST_KEY', 'first_name', 'middle_name', 'last_name', 'house_number',
       'street_address', 'city', 'state', 'zip_code', 'zest_in_state_fips',
       'ZEST_KEY_COL', 'FROMHN_LEFT', 'TOHN_LEFT', 'GEOID_CT_zrp',
       'GEOID_BG_zrp', 'GEOID_ZIP', 'GEOID', 'GEOID_CT_census',
       'GEOID_BG_census', 'state_fips', 'county_fips', 'tract', 'CT_match',
       'ct_state_match', 'ct_county_match', 'ct_tract_match'],
      dtype='object')
True     0.513458
False    0.486542
Name: CT_match, dtype: float64


,house_number,street_address,city,state,zip_code,GEOID_CT_census,GEOID_CT_zrp,ct_state_match,ct_county_match,ct_tract_match
0,1974,WEBSTER GROVE DR,MEBANE,NC,27302,37001021204,37001021204,True,True,True
1,774,BAKER CT,HAW RIVER,NC,27258,37001021206,37001021206,True,True,True
2,142,MONTREE LN,GRAHAM,NC,27253,37001021102,37001021102,True,True,True
4,1739,PETTY RD,GRAHAM,NC,27253,37001021901,37105030601,True,False,False
5,1422,KNOLLWOOD DR,BURLINGTON,NC,27217,37001020301,37183920800,True,False,False
6,2406,OAKWOOD DR,BURLINGTON,NC,27215,37001020601,37071010800,True,False,False
7,<NA,203C EARL KIMBER RD,BURLINGTON,NC,27217,37001021400,37001021400,True,True,True
11,2008,S MEBANE ST 733E,BURLINGTON,NC,27215,37001020702,37001020701,True,True,False
12,457,CHEEKS LN,GRAHAM,NC,27253,37001021102,37001021102,True,True,True
13,2466,BARBER RD K,ELON,NC,27244,37001021600,37125970100,True,False,False
